# M2 - IA Agética - Aprimorando a Geração de SQL com Reflexão

## 1. Introdução
### 1.1. Visão geral do laboratório

Neste laboratório, você explorará como o padrão de reflexão pode aprimorar um fluxo de trabalho agético que converte perguntas em linguagem natural em consultas de banco de dados escritas em SQL. Você verá como um agente pode identificar problemas em suas próprias saídas, refiná-las e aprimorar sua resposta antes de fornecer uma resposta final.

### 🎯 1.2 Resultado de aprendizagem

Você praticará a aplicação do padrão de reflexão para aprimorar a capacidade de um fluxo de trabalho agético de escrever consultas SQL. Para isso, você codificará uma etapa de reflexão no fluxo de trabalho, para que o agente:

* Revise seus próprios resultados intermediários (como rascunhos de SQL ou saídas de ferramentas).

* Identifique erros ou lacunas.

* Verifique suas respostas e o uso de ferramentas.

* Refine a saída antes de enviar a resposta final.

## 2. Configuração: Inicializar o Ambiente e o Cliente

Nesta etapa, você preparará seu espaço de trabalho para começar a programar imediatamente. Você deverá:

1. **Importar as bibliotecas principais do Python**

* `json` para lidar com dados estruturados.

* `pandas` para trabalhar com dados tabulares.

* `dotenv` para carregar variáveis ​​de ambiente (por exemplo, chaves de API).

2. **Carregar as variáveis ​​de ambiente**

Isso garante que seu espaço de trabalho esteja configurado corretamente com as chaves e configurações necessárias.

3. **Importar o módulo `utils.py`**

Este arquivo contém funções auxiliares que você usará para formatar as saídas e dar suporte às etapas posteriores do fluxo de trabalho.

**Observação:** Se você quiser explorar o conteúdo de `utils.py`, vá ao menu superior e selecione **Arquivo > Abrir**.

In [ ]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

### 2.1 Primeiros passos com o AISuite

Os laboratórios deste curso utilizarão um pacote chamado aisuite [repositório aisuite](https://github.com/andrewyng/aisuite) que facilita a chamada de LLMs hospedados por diferentes provedores de modelos. Andrew abordará mais detalhes sobre o aisuite no Módulo 3. Agora, inicialize o `cliente aisuite`. Este cliente oferece uma maneira única e unificada de se conectar e interagir com diferentes LLMs — assim, você não precisa se preocupar com a configuração individual de cada modelo.

In [ ]:
import aisuite as ai

client = ai.Client()

### 2.2. Configurar o Banco de Dados

Nesta etapa, você criará um banco de dados SQLite local chamado **`products.db`**.

O banco de dados será preenchido automaticamente com dados de produtos gerados aleatoriamente.

Você usará esses dados posteriormente no laboratório para praticar e testar suas consultas SQL.

In [ ]:
utils.create_transactions_db()

Você pode inspecionar o esquema da tabela executando a célula abaixo.

In [ ]:
utils.print_html(utils.get_schema('products.db'))

Nesta tabela, cada linha representa um **evento** (inserção, reposição de estoque, venda ou atualização de preço). Análises como níveis de estoque, vendas ou tendências de preços são derivadas desses eventos.

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
<strong>🔎 Visão geral do esquema:</strong><br><br>
• <code>id</code> → ID exclusivo do evento (autoincremento).<br>
• <code>product_id</code>, <code>product_name</code>, <code>brand</code>, <code>category</code>, <code>color</code> → identifica o produto.<br>
• <code>action</code> → tipo de evento (<em>insert</em>, <em>restock</em>, <em>sale</em>, <em>price_update</em>).<br>
• <code>qty_delta</code> → variação de estoque (+ para inserção/reabastecimento, – para venda, 0 para atualizações de preço).<br>
• <code>unit_price</code> → preço naquele momento (NULL para reabastecimento).<br>
• <code>notes</code> → descrição opcional do evento.<br>
• <code>ts</code> → registro de data e hora em que o evento foi registrado.<br>
</div>

> Com este esquema, você sempre pode reconstruir o **estado atual** (estoque, preço mais recente, vendas totais) agregando o histórico de eventos.

## 3. Build a SQL generator

### 3.1. Usar um LLM para consultar um banco de dados

Nesta etapa, você usará uma função que transforma suas perguntas em linguagem natural em consultas SQL.

Você fornece sua pergunta e o esquema do banco de dados como entrada. O LLM então gera a consulta SQL que responde à sua pergunta.

Dessa forma, **você** pode se concentrar em fazer perguntas enquanto o modelo se encarrega de escrever a consulta.

In [ ]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

Execute a célula abaixo para ver como a função **`generate_sql`** cria a **primeira versão (V1)** de uma consulta SQL para a tabela `transactions`, partindo de uma pergunta em linguagem natural.
Basta fornecer a **pergunta**, o **esquema** e o **nome do modelo**, e a função retornará o rascunho inicial da consulta.

In [ ]:
# Example usage of generate_sql

# We provide the schema as a string
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# We ask a question about the data in natural language
question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")

# Generate the SQL query using the specified model
sql_V1 = generate_sql(question, schema, model="openai:gpt-4.1")

# Display the generated SQL query
utils.print_html(sql_V1, title="SQL Query V1")


#### 3.1.1. Execução da Consulta

Agora você executará a **primeira versão (V1) da consulta SQL** e inspecionará seus resultados. Esta etapa é importante porque permite verificar se a consulta gerada pelo LLM realmente recupera as informações esperadas do banco de dados.

- **`utils.execute_sql(...)`**: executa a consulta SQL gerada (V1) no banco de dados `products.db` e retorna a saída como um DataFrame do pandas. Trabalhar com DataFrames facilita a inspeção, a análise e a passagem dos resultados para as etapas posteriores do fluxo de trabalho.

- **`utils.print_html(...)`**: pega o DataFrame e o renderiza como uma tabela HTML formatada no notebook. Isso torna a saída bruta mais legível e ajuda a identificar rapidamente se o resultado da consulta está alinhado com a pergunta do usuário.

In [ ]:
# Execute a consulta SQL gerada (sql_V1) no banco de dados products.db.
# O resultado é retornado como um DataFrame do pandas.
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# Renderize o DataFrame como uma tabela HTML no notebook.
# Isso facilita a leitura e a interpretação do resultado da consulta.
utils.print_html(df_sql_V1, title="Output of SQL Query V1 - ❌ Does NOT fully answer the question")


<div style="border:1px solid #ef4444; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ <strong>Problema de Saída V1:</strong> O resultado da consulta não responde completamente à pergunta, pois o valor total de vendas é <em>negativo</em> (`-190571.46`).

<br><br>
Neste conjunto de dados, um <code>qty_delta</code> é registrado como um **número negativo para vendas** (saída de estoque) e como um **número positivo para devoluções ou reabastecimentos** (entrada de estoque). A consulta usou `SUM(qty_delta)`, portanto, ao somar as transações, os valores negativos predominam e produzem um total negativo.

<br>
Isso significa que o SQL é tecnicamente válido, mas semanticamente incorreto — o total de vendas não deve ser expresso como um valor negativo.

<br>
➡️ Este problema <strong>evidencia a necessidade de reflexão</strong>: o modelo deve refinar sua lógica (por exemplo, multiplicando `qty_delta` por `-1` ao calcular as vendas ou aplicando `ABS()`) para que a consulta final reflita os totais de vendas reais.

</div>

### 3.2. Aprimorando Consultas SQL com Reflexão

Nesta seção, você aprenderá como **aprimorar consultas SQL usando reflexão**.

Primeiro, o LLM pode revisar apenas o **texto SQL** em relação à pergunta e ao esquema, e propor melhorias, se necessário.

Em seguida, o LLM também pode considerar os **resultados reais da execução da consulta** para detectar problemas sutis, como totais negativos, filtros ausentes ou erros de agrupamento.

Juntas, essas abordagens mostram como a reflexão torna seu fluxo de trabalho SQL mais **confiável e preciso** — primeiro verificando a lógica no papel e depois validando-a com dados reais.

#### 3.2.1. Primeira Tentativa: Refinar uma consulta SQL

Nesta função, você solicita a um LLM que **revise** uma consulta SQL em relação à pergunta original e ao esquema (por exemplo, como o definido na seção 3.1). O modelo reflete sobre se a consulta responde completamente à pergunta e, caso contrário, propõe uma versão aprimorada.

* **Entradas**:

* a **pergunta** do usuário

* a **consulta SQL original**

* o **esquema da tabela**

* **Saídas**:

* **feedback** → uma breve avaliação (por exemplo, “válida, mas falta um filtro de data”)

* **sql_refinado** → o SQL final (inalterado se correto, ou atualizado se forem necessárias melhorias)

Esta função **não executa SQL**. Ela apenas inspeciona a consulta e sugere refinamentos quando a lógica não corresponde perfeitamente à intenção.

In [ ]:
def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Reflect on whether a query's *shown output* answers the question,
    and propose an improved SQL if needed.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
You are a SQL reviewer and refiner.

User asked:
{question}

Original SQL:
{sql_query}

Table Schema:
{schema}

Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
Step 2: If improvement is needed, provide a refined SQL query for SQLite.
If the original SQL is already correct, return it unchanged.

Return STRICT JSON with two fields:
{{
  "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
  "refined_sql": "<final SQL to run>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # Fallback if model doesn't return valid JSON
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql


Execute a célula a seguir para gerar a **consulta SQL refinada (V2)**. Esta etapa irá:

* Exibir a **consulta SQL inicial (V1)** gerada na Seção 3.1 para a pergunta *“Qual cor de produto tem o maior volume de vendas?”*
* Mostrar o **feedback** do modelo e sua **proposta de SQL refinada (V2)**.

* Executar a **consulta SQL original (V1)** no banco de dados e apresentar sua **saída real**, para que você possa ver por que o refinamento foi necessário.

In [ ]:
# Example: refine the generated SQL (V1 → V2)

feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,   # <- comes from generate_sql() (V1)
    schema=schema, # <- we reuse the schema from section 3.1
    model="openai:gpt-4.1"
)

# Display the original prompt
utils.print_html(question, title="User Question")

# --- V1 ---
utils.print_html(sql_V1, title="Generated SQL Query (V1)")

# Execute and show V1 output
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback + V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and show V2 output
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 - ❌ Does NOT fully answer the question")

<div style="border:1px solid #fecaca; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ **Como você pode ver...**
Embora o SQL gerado parecesse correto e o feedback o confirmasse, os **resultados reais da consulta** mostram um valor negativo para `total_sales`.

Como mencionado anteriormente, isso ocorreu porque o SQL multiplicou `qty_delta * unit_price` sem considerar que os eventos de vendas armazenam **quantidades negativas** (`qty_delta < 0`). A inversão de sinal (+ vs –) é uma questão semântica sutil que nem sempre pode ser detectada apenas pela análise do texto da consulta.

👉 É por isso que o agente também deve refletir sobre a **saída da execução** — para detectar problemas como sinais incorretos, filtros ausentes ou agregações incorretas. O feedback externo dos resultados da consulta fundamenta o processo de refinamento na realidade, e não apenas na estrutura do SQL.

</div>

#### 3.2.2. Abordagem Final: Refinando uma Consulta SQL com Feedback Externo

Ao contrário da etapa anterior, em que o modelo apenas **analisava o texto SQL**, agora você fornecerá os **resultados reais da execução da consulta** como feedback externo.

Esse feedback vem da execução da consulta SQL no banco de dados — assim como no exemplo em vídeo de Andrew — para que o LLM possa usar a saída real para avaliar se a consulta realmente responde à pergunta.

Nesta etapa, você receberá:

* Uma breve avaliação da sua consulta, com base na **saída real da execução**.

* Sugestões concretas para melhorias (por exemplo, filtros ausentes, problemas de agrupamento, erros de sinal).

* Uma instrução SQL refinada que corresponda melhor à pergunta original.

In [ ]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Evaluate whether the SQL result answers the user's question and,
    if necessary, propose a refined version of the query.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )

    
    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # Fallback if the model does not return valid JSON:
        # use the raw content as feedback and keep the original SQL
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

Execute a célula a seguir para ver como o **feedback externo** dos resultados da consulta melhora o refinamento do SQL.

Neste exemplo, você irá:

* Exibir a **consulta SQL original (V1)** gerada a partir da sua pergunta.

* Mostrar a **saída de V1**, que destaca por que a tentativa inicial não responde completamente à pergunta.

* Fornecer **feedback** do LLM com base nessa saída.

* Apresentar a **consulta SQL refinada (V2)** que resolve o problema.

* Executar **V2** e exibir sua saída, confirmando que agora ela ✅ responde completamente à pergunta.

In [ ]:
# Example: Refine SQL with External Feedback (V1 → V2)

# Execute the original SQL (V1)
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# Use external feedback to evaluate and refine
feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,   # V1 query
    df_feedback=df_sql_V1,    # Output of V1
    schema=schema,
    model="openai:gpt-4.1"
)

# --- V1 ---
utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback & V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and display V2 results
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")


<div style="border:1px solid #bbf7d0; border-left:6px solid #22c55e; background:#dcfce7; border-radius:6px; padding:12px 14px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

✅ **Sucesso!**
Com o SQL refinado, o problema com valores negativos de `qty_delta` foi corrigido aplicando `ABS(qty_delta)` (ou equivalentemente `-qty_delta` para vendas).

Agora a saída é **positiva e significativa**, mostrando a cor correta com o maior total de vendas. Isso demonstra a importância de combinar **reflexão + feedback externo**:
- O texto SQL sozinho parecia correto.

- A saída da execução revelou um erro de sinal.
A consulta refinada corrigiu a lógica, levando a uma resposta correta.

</div>

### 3.3. Unindo tudo — Construindo o Fluxo de Trabalho de Consulta ao Banco de Dados

Nesta etapa, **você** usará uma função que automatiza todo o fluxo de trabalho de criação, execução e aprimoramento de consultas SQL com um LLM.

O fluxo de trabalho percorre estas etapas principais:

1. **Extrair o esquema do banco de dados**
2. **Gerar uma consulta SQL inicial (V1)** a partir da sua pergunta em linguagem natural
3. **Refletir sobre a V1 usando o feedback da execução** — revisar os resultados da consulta e refinar o SQL, se necessário
4. **Executar a consulta SQL aprimorada (V2)** para garantir que ela responda completamente à sua pergunta

Ao final, **você** verá:

* As consultas inicial e refinada
* Seus respectivos resultados
* Feedback do LLM explicando o refinamento

Isso facilita o trabalho com consultas SQL de forma mais fluida, precisa e totalmente automatizada.

In [ ]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
    """
    End-to-end workflow to generate, execute, evaluate, and refine SQL queries.

    Steps:
      1) Extract database schema
      2) Generate SQL (V1)
      3) Execute V1 → show output
      4) Reflect on V1 with execution feedback → propose refined SQL (V2)
      5) Execute V2 → show final answer
    """

    # 1) Schema
    schema = utils.get_schema(db_path)
    utils.print_html(
        schema,
        title="📘 Step 1 — Extract Database Schema"
    )

    # 2) Generate SQL (V1)
    sql_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(
        sql_v1,
        title="🧠 Step 2 — Generate SQL (V1)"
    )

    # 3) Execute V1
    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(
        df_v1,
        title="🧪 Step 3 — Execute V1 (SQL Output)"
    )

    # 4) Reflect on V1 with execution feedback → refine to V2
    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,          # external feedback: real output of V1
        schema=schema,
        model=model_evaluation,
    )
    utils.print_html(
        feedback,
        title="🧭 Step 4 — Reflect on V1 (Feedback)"
    )
    utils.print_html(
        sql_v2,
        title="🔁 Step 4 — Refined SQL (V2)"
    )

    # 5) Execute V2
    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(
        df_v2,
        title="✅ Step 5 — Execute V2 (Final Answer)"
    )


### 3.4. Executando o Fluxo de Trabalho SQL

Agora é a sua vez de executar o pipeline completo de processamento SQL. Você pode experimentar diferentes combinações dos seguintes modelos OpenAI, cada um com diferentes capacidades e desempenho:

* `openai:gpt-4o`
* `openai:gpt-4.1`
* `openai:gpt-4.1-mini`
* `openai:gpt-3.5-turbo`

💡 Neste fluxo de trabalho, o modelo `openai:gpt-4.1` geralmente apresenta os melhores resultados para tarefas de autorreflexão.

**Importante:** Como os Modelos de Linguagem de Grande Porte (LLMs) são estocásticos, cada execução pode retornar resultados ligeiramente diferentes.

Recomendamos que você experimente diferentes modelos e combinações para encontrar a configuração ideal para você.

In [ ]:
run_sql_workflow(
    "products.db", 
    "Which color of product has the highest total sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

## 4. Conclusões Finais

Ao concluir este laboratório, você aprendeu a:

* Usar um LLM para transformar perguntas em linguagem natural em consultas SQL.

* Aplicar **padrões de reflexão** (com e sem feedback externo) para aprimorar o SQL gerado.

* Automatizar um fluxo de trabalho SQL completo, desde a extração do esquema até a execução e o refinamento da consulta.

* Experimentar diferentes modelos de LLM para comparar desempenho e precisão.

A principal conclusão é que a reflexão torna seu agente mais confiável: em vez de parar na primeira tentativa, o agente pode revisar, aprimorar e fornecer resultados que correspondam melhor à sua intenção.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Parabéns!</strong>

Você concluiu o laboratório sobre a construção de um **fluxo de trabalho SQL ativo (V1 → V2)**.

Ao longo do processo, **você** praticou como a reflexão, a execução e a validação se unem em um pipeline confiável.

Você também percebeu a importância do **feedback externo**: às vezes, o texto SQL sozinho parece correto, mas a saída da execução revela problemas ocultos (como totais negativos, filtros ausentes ou erros de agrupamento).

Com essas habilidades, você agora está pronto para projetar seus próprios **pipelines de agentes** — fluxos de trabalho que:
* Geram consultas iniciais (V1).

* Refletem e refinam essas consultas com feedback de execução (V2).

* Fornecem resultados mais seguros, transparentes e confiáveis. 🌟

</div>